In [4]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy tqdm matplotlib

In [5]:
import os, json, math, random
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


Matplotlib is building the font cache; this may take a moment.


'cuda'

In [10]:
# Project-relative paths (adjust if needed)
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
SAVE_DIR = Path("../saved_model")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Expecting:
#  - LIAR TSVs in RAW_DIR: liar_train.tsv, liar_valid.tsv, liar_test.tsv
#  - FakeNewsNet CSV in RAW_DIR: fakenewsnet.csv (columns: text, label in {fake, real} or {0,1})

BASE_MODEL = "distilbert-base-uncased"  # good balance of speed/quality
MAX_LEN = 160
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
LABEL2ID = {"fake": 0, "real": 1}
ID2LABEL = {v:k for k,v in LABEL2ID.items()}


In [14]:
# Helper to map LIAR multi-class to binary
FAKE_SET = {"pants-fire", "pants-on-fire", "false", "barely-true"}
REAL_SET = {"half-true", "mostly-true", "true"}

def load_liar_split(tsv_path: Path) -> pd.DataFrame:
    cols = [
        "id","label","statement","subject","speaker","job_title","state",
        "party","barely_true","false","half_true","mostly_true","pants_on_fire","context"
    ]
    df = pd.read_csv(tsv_path, sep="\t", header=None, names=cols, quoting=3, on_bad_lines="skip")
    df["label_norm"] = df["label"].str.lower().str.replace("_","-").str.replace(" ", "-", regex=False)
    df = df[df["label_norm"].isin(FAKE_SET.union(REAL_SET))].copy()
    df["binary_label"] = df["label_norm"].apply(lambda x: "fake" if x in FAKE_SET else "real")
    out = df[["statement", "binary_label"]].rename(columns={"statement":"text", "binary_label":"label"})
    out["text"] = out["text"].astype(str).str.strip()
    out = out[out["text"].str.len() > 0]
    return out

liar_train = load_liar_split(RAW_DIR / "liar_train.tsv")
liar_valid = load_liar_split(RAW_DIR / "liar_valid.tsv")
liar_test  = load_liar_split(RAW_DIR / "liar_test.tsv")

liar_all = pd.concat([liar_train, liar_valid, liar_test], ignore_index=True).drop_duplicates(subset=["text"])
liar_all["label"] = liar_all["label"].map(LABEL2ID)
liar_all.head(), liar_all["label"].value_counts()


(                                                text  label
 0  Says the Annies List political group supports ...      0
 1  When did the decline of coal start? It started...      1
 2  Hillary Clinton agrees with John McCain "by vo...      1
 3  Health care reform legislation is likely to ma...      0
 4  The economic turnaround started at the end of ...      1,
 label
 1    7156
 0    5654
 Name: count, dtype: int64)

In [21]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
df = pd.read_csv(RAW_DIR / "gossipcop_fake.csv")
print(df.columns)



Index(['id', 'news_url', 'title', 'tweet_ids'], dtype='object')


In [22]:
fakenews_files = [
    "gossipcop_fake.csv",
    "gossipcop_real.csv",
    "politifact_fake.csv",
    "politifact_real.csv"
]

dfs = []
for f in fakenews_files:
    df = pd.read_csv(RAW_DIR / f)
    df["label"] = 0 if "fake" in f else 1
    # Rename 'title' to 'text' since 'title' contains the news content
    df = df.rename(columns={"title":"text"})
    dfs.append(df)

fakenewsnet = pd.concat(dfs, ignore_index=True)
fakenewsnet = fakenewsnet[["text","label"]]

fakenewsnet.to_csv(RAW_DIR / "fakenewsnet.csv", index=False)


In [23]:
# Load preprocessed FakeNewsNet
fakenewsnet = pd.read_csv(RAW_DIR / "fakenewsnet.csv")

# Load LIAR splits (adjust path to your raw LIAR files)
liar_train = pd.read_csv(RAW_DIR / "liar_train.tsv", sep="\t", header=None)
liar_valid = pd.read_csv(RAW_DIR / "liar_valid.tsv", sep="\t", header=None)
liar_test  = pd.read_csv(RAW_DIR / "liar_test.tsv", sep="\t", header=None)

# Keep only necessary columns for LIAR
liar_train = liar_train[[2,1]].rename(columns={2:"text", 1:"label"})
liar_valid = liar_valid[[2,1]].rename(columns={2:"text", 1:"label"})
liar_test  = liar_test[[2,1]].rename(columns={2:"text", 1:"label"})

# Normalize LIAR labels: convert textual labels to 0/1 (fake/real)
fake_set = ["pants-on-fire", "false", "barely-true", "half-true"]
real_set = ["mostly-true", "true"]

for df in [liar_train, liar_valid, liar_test]:
    df["label"] = df["label"].str.lower().str.replace("_","-")
    df["label"] = df["label"].apply(lambda x: 0 if x in fake_set else 1 if x in real_set else None)
    df.dropna(inplace=True)

# Combine LIAR train + FakeNewsNet for final training dataset
train_data = pd.concat([liar_train, fakenewsnet], ignore_index=True)

# Preprocess text: lowercase & strip spaces
for df in [train_data, liar_valid, liar_test]:
    df["text"] = df["text"].str.lower().str.strip()

# Quick check
print("Training data shape:", train_data.shape)
print("Validation data shape:", liar_valid.shape)
print("Test data shape:", liar_test.shape)

train_data.head()


Training data shape: (32597, 2)
Validation data shape: (1168, 2)
Test data shape: (1175, 2)


,text,label
0,says the annies list political group supports ...,0.0
1,when did the decline of coal start? it started...,0.0
2,"hillary clinton agrees with john mccain ""by vo...",1.0
3,health care reform legislation is likely to ma...,0.0
4,the economic turnaround started at the end of ...,0.0


In [24]:
!pip install -q transformers datasets torch scikit-learn

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report


In [25]:
MODEL_NAME = "distilbert-base-uncased"  # small and fast model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [26]:
MAX_LEN = 128  # maximum token length

def tokenize_texts(texts):
    return tokenizer(
        texts.tolist(), 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_LEN, 
        return_tensors="pt"
    )

train_encodings = tokenize_texts(train_data["text"])
valid_encodings = tokenize_texts(liar_valid["text"])
test_encodings  = tokenize_texts(liar_test["text"])


In [33]:
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        # Ensure labels are long integers for CrossEntropyLoss
        self.labels = torch.tensor(labels.values, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# Create datasets
train_dataset = FakeNewsDataset(train_encodings, train_data["label"])
valid_dataset = FakeNewsDataset(valid_encodings, liar_valid["label"])
test_dataset  = FakeNewsDataset(test_encodings,  liar_test["label"])


In [34]:
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)


In [35]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # binary classification: fake or real
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [36]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
print("Using device:", device)


Using device: cuda


In [37]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)


In [ ]:
from tqdm import tqdm
from torch.nn import CrossEntropyLoss

EPOCHS = 3
loss_fn = CrossEntropyLoss()  # explicit, but optional (HF uses this internally for num_labels>1)

for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_loader, leave=True)
    total_loss = 0
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)  # shape [batch_size], dtype long
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        # HF model automatically uses CrossEntropyLoss for num_labels>1
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())
    
    print(f"Average loss for epoch {epoch+1}: {total_loss/len(train_loader):.4f}")
